<a href="https://colab.research.google.com/github/Abishek9342/waveform/blob/main/Montreal_Forced_Aligner_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()


!conda install -c conda-forge montreal-forced-aligner -y
!conda install -c conda-forge textgrid -y


✨🍰✨ Everything looks OK!
Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ | / - \ | / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.

Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - \ | / - \ | / - \ | / done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [2]:
from google.colab import drive, files
import os
import sys
from pathlib import Path
import subprocess
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')

# Project paths
project_dir = '/content/drive/My Drive/MFA_Implementation'
data_dir = f'{project_dir}/data'
audio_dir = f'{data_dir}/audio'
text_dir = f'{data_dir}/text'
lexicon_dir = f'{data_dir}/lexicon'
output_dir = f'{project_dir}/output'
results_dir = f'{output_dir}/results'
plots_dir = f'{output_dir}/visualizations'
reports_dir = f'{output_dir}/reports'

# Create directories
for directory in [audio_dir, text_dir, lexicon_dir, results_dir, plots_dir, reports_dir]:
    Path(directory).mkdir(parents=True, exist_ok=True)

print(f"\n Project directories created at:")
print(f"  Location: {project_dir}")
print(f"\n Structure:")
print(f"  ├── data/")
print(f"  │   ├── audio/")
print(f"  │   ├── text/")
print(f"  │   └── lexicon/")
print(f"  └── output/")
print(f"      ├── results/")
print(f"      ├── visualizations/")
print(f"      └── reports/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 Project directories created at:
  Location: /content/drive/My Drive/MFA_Implementation

 Structure:
  ├── data/
  │   ├── audio/
  │   ├── text/
  │   └── lexicon/
  └── output/
      ├── results/
      ├── visualizations/
      └── reports/


In [3]:


!mfa model download acoustic english_us_arpa

print("\n✓ Models downloaded successfully")
print("  Models location: ~/.mfa/models/")

Local version of model already exists (/root/Documents/MFA/pretrained_models/acoustic/english_us_arpa.zip). Use the --ignore_cache flag to force redownloading.

✓ Models downloaded successfully
  Models location: ~/.mfa/models/


In [5]:
# Check if directories exist and have files
print("Checking directories...")
print(f"Audio dir: {audio_dir}")
print(f"Audio files: {list(Path(audio_dir).glob('*.wav'))}")
print(f"\nText dir: {text_dir}")
print(f"Text files: {list(Path(text_dir).glob('*.txt'))}")

# If empty, recreate sample data
if not list(Path(audio_dir).glob('*.wav')):
    print("\n Creating sample data...")
    import wave

    sample_audios = {
        'sample001': "hello world how are you today",
        'sample002': "this is a test of the alignment system",
        'sample003': "montreal forced aligner using kaldi"
    }

    for audio_id, transcription in sample_audios.items():
        audio_path = f'{audio_dir}/{audio_id}.wav'
        text_path = f'{text_dir}/{audio_id}.txt'

        # Create audio
        num_samples = 3 * 16000
        audio_data = np.zeros(num_samples, dtype=np.int16)
        with wave.open(audio_path, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(16000)
            wav_file.writeframes(audio_data.tobytes())

        # Create text
        with open(text_path, 'w') as f:
            f.write(transcription)

        print(f"✓ Created {audio_id}")

    print("✓ Sample data recreated!")

Checking directories...
Audio dir: /content/drive/My Drive/MFA_Implementation/data/audio
Audio files: [PosixPath('/content/drive/My Drive/MFA_Implementation/data/audio/sample001.wav'), PosixPath('/content/drive/My Drive/MFA_Implementation/data/audio/sample002.wav'), PosixPath('/content/drive/My Drive/MFA_Implementation/data/audio/sample003.wav'), PosixPath('/content/drive/My Drive/MFA_Implementation/data/audio/demo001.wav'), PosixPath('/content/drive/My Drive/MFA_Implementation/data/audio/demo002.wav'), PosixPath('/content/drive/My Drive/MFA_Implementation/data/audio/demo003.wav')]

Text dir: /content/drive/My Drive/MFA_Implementation/data/text
Text files: [PosixPath('/content/drive/My Drive/MFA_Implementation/data/text/sample002.txt'), PosixPath('/content/drive/My Drive/MFA_Implementation/data/text/sample003.txt'), PosixPath('/content/drive/My Drive/MFA_Implementation/data/text/sample001.txt'), PosixPath('/content/drive/My Drive/MFA_Implementation/data/text/demo001.txt'), PosixPath('/

In [6]:


# Check audio files
audio_files = list(Path(audio_dir).glob('*.wav'))
print(f"\n✓ Audio Files Found: {len(audio_files)}")
for audio_file in audio_files:
    print(f"   - {audio_file.name}")

# Check text files
text_files = list(Path(text_dir).glob('*.txt'))
print(f"\n✓ Text Files Found: {len(text_files)}")
for text_file in text_files:
    with open(text_file, 'r') as f:
        content = f.read().strip()
    print(f"   - {text_file.name}: '{content}'")

# Validate matching pairs
audio_names = {f.stem for f in audio_files}
text_names = {f.stem for f in text_files}
matched = audio_names & text_names
unmatched = audio_names ^ text_names

print(f"\n✓ Matching pairs: {len(matched)}")
if unmatched:
    print(f"  Unmatched files: {unmatched}")
else:
    print("   All audio files have corresponding text files ✓")


✓ Audio Files Found: 6
   - sample001.wav
   - sample002.wav
   - sample003.wav
   - demo001.wav
   - demo002.wav
   - demo003.wav

✓ Text Files Found: 6
   - sample002.txt: 'this is a test of the alignment system'
   - sample003.txt: 'montreal forced aligner using kaldi'
   - sample001.txt: 'hello world how are you today'
   - demo001.txt: 'hello world how are you today'
   - demo002.txt: 'this is a test of the alignment system'
   - demo003.txt: 'montreal forced aligner using kaldi'

✓ Matching pairs: 6
   All audio files have corresponding text files ✓


In [7]:
# Verify directories and files exist
print("Checking paths...")
print(f"Audio directory: {audio_dir}")
print(f"Text directory: {text_dir}")

# List what's in them
audio_files = list(Path(audio_dir).glob('*.wav'))
text_files = list(Path(text_dir).glob('*.txt'))

print(f"\n✓ Audio files found: {len(audio_files)}")
for f in audio_files:
    print(f"  - {f.name}")

print(f"\n✓ Text files found: {len(text_files)}")
for f in text_files:
    print(f"  - {f.name}")

if len(audio_files) == 0 or len(text_files) == 0:
    print("\n⚠️ Files missing! Running sample data creation...")


Checking paths...
Audio directory: /content/drive/My Drive/MFA_Implementation/data/audio
Text directory: /content/drive/My Drive/MFA_Implementation/data/text

✓ Audio files found: 6
  - sample001.wav
  - sample002.wav
  - sample003.wav
  - demo001.wav
  - demo002.wav
  - demo003.wav

✓ Text files found: 6
  - sample002.txt
  - sample003.txt
  - sample001.txt
  - demo001.txt
  - demo002.txt
  - demo003.txt


In [8]:
import wave
import numpy as np
from pathlib import Path
import shutil


# Use a simpler corpus structure that MFA prefers
corpus_dir = '/root/MFA_corpus'
Path(corpus_dir).mkdir(exist_ok=True)

# Remove old files if they exist
for f in Path(corpus_dir).glob('*'):
    if f.is_file():
        f.unlink()

print("\nCreating MFA corpus structure...\n")

# Create audio and text files in the same directory
test_data = {
    'speaker1_utt1': ("hello world how are you today", 440),
    'speaker1_utt2': ("this is a test of alignment", 480),
    'speaker1_utt3': ("montreal forced aligner", 520)
}

for utterance_id, (transcription, freq) in test_data.items():
    # Create audio file
    audio_path = f'{corpus_dir}/{utterance_id}.wav'
    text_path = f'{corpus_dir}/{utterance_id}.txt'

    sample_rate = 16000
    duration = 3
    num_samples = duration * sample_rate
    t = np.linspace(0, duration, num_samples)

    # Create tone with variation
    audio_data = (np.sin(2 * np.pi * freq * t) * 20000).astype(np.int16)

    # Add amplitude variation
    envelope = np.linspace(1, 0.5, num_samples)
    audio_data = (audio_data * envelope).astype(np.int16)

    # Write WAV
    with wave.open(audio_path, 'wb') as f:
        f.setnchannels(1)
        f.setsampwidth(2)
        f.setframerate(sample_rate)
        f.writeframes(audio_data.tobytes())

    # Write text
    with open(text_path, 'w') as f:
        f.write(transcription)

    print(f"✓ {utterance_id}")
    print(f"  Audio: {audio_path}")
    print(f"  Text: {text_path}")
    print(f"  Transcription: '{transcription}'")

print(f"\n✓ Corpus ready at: {corpus_dir}")
print(f"✓ Files in corpus: {len(list(Path(corpus_dir).glob('*.wav')))}")

# Update paths for alignment
corpus_dir = '/root/MFA_corpus'
results_dir = '/root/MFA_output'
Path(results_dir).mkdir(exist_ok=True)


Creating MFA corpus structure...

✓ speaker1_utt1
  Audio: /root/MFA_corpus/speaker1_utt1.wav
  Text: /root/MFA_corpus/speaker1_utt1.txt
  Transcription: 'hello world how are you today'
✓ speaker1_utt2
  Audio: /root/MFA_corpus/speaker1_utt2.wav
  Text: /root/MFA_corpus/speaker1_utt2.txt
  Transcription: 'this is a test of alignment'
✓ speaker1_utt3
  Audio: /root/MFA_corpus/speaker1_utt3.wav
  Text: /root/MFA_corpus/speaker1_utt3.txt
  Transcription: 'montreal forced aligner'

✓ Corpus ready at: /root/MFA_corpus
✓ Files in corpus: 3


In [9]:


start_time = datetime.now()
print(f"\nAlignment started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("Processing audio files...")

# Use the simpler corpus directory
result = subprocess.run(
    f'mfa align "{corpus_dir}" english_us_arpa english_us_arpa "{results_dir}"',
    shell=True,
    capture_output=True,
    text=True,
    timeout=600
)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(result.stdout)
if result.returncode != 0:
    print("⚠️ Alignment stderr:")
    print(result.stderr[-800:])

# List output files
output_files = list(Path(results_dir).glob('*.TextGrid'))
print(f"\n✓ Alignment completed in {duration:.2f} seconds")
print(f"✓ Generated {len(output_files)} TextGrid files:")
for output_file in output_files:
    print(f"   - {output_file.name}")


Alignment started: 2026-04-18 07:19:23
Processing audio files...
  33% ━━━━━━━━━━━━╸                          1/3  [ 0:00:02 < -:--:-- , ? it/s ]
  33% ━━━━━━━━━━━━╸                          1/3  [ 0:00:02 < -:--:-- , ? it/s ]


✓ Alignment completed in 10.99 seconds
✓ Generated 1 TextGrid files:
   - speaker1_utt3.TextGrid


In [ ]:
# Force reinstall and restart kernel
!pip install --force-reinstall --no-cache-dir textgrid praat-textgrids

# Restart the kernel
import os
os.kill(os.getpid(), 9)

  Preparing metadata (setup.py) ... done
  Created wheel for textgrid: filename=TextGrid-1.6.1-py3-none-any.whl size=10147 sha256=17a5953bfa174670ca0ef5d2eecd2db2c5a128b9d2b03fc99fb1b1364dd20df0
  Stored in directory: /tmp/pip-ephem-wheel-cache-r8hdba9n/wheels/7a/c5/96/5e43aa4c640995fbbb0b9a7b98e6007bfd777add3c7e56d70a
Successfully built textgrid
  Attempting uninstall: textgrid
    Found existing installation: TextGrid 1.6.1
    Uninstalling TextGrid-1.6.1:
      Successfully uninstalled TextGrid-1.6.1
  Attempting uninstall: praat-textgrids
    Found existing installation: praat-textgrids 1.4.0
    Uninstalling praat-textgrids-1.4.0:
      Successfully uninstalled praat-textgrids-1.4.0


In [1]:
from pathlib import Path
import pandas as pd


def parse_textgrid_file(textgrid_path):
    """Parse TextGrid file manually without textgrid module"""
    try:
        with open(textgrid_path, 'r') as f:
            content = f.read()

        alignments = {
            'words': [],
            'phones': [],
        }

        lines = content.split('\n')
        current_tier = None
        i = 0

        while i < len(lines):
            line = lines[i].strip()

            # Detect tier name
            if 'name = "words"' in line:
                current_tier = 'words'
            elif 'name = "phones"' in line:
                current_tier = 'phones'

            # Parse intervals
            if 'xmin = ' in line and current_tier:
                try:
                    xmin = float(line.split('=')[1].strip())
                    xmax = float(lines[i+1].split('=')[1].strip())
                    text = lines[i+2].split('=')[1].strip().strip('"')

                    if text:  # Only add non-empty
                        entry = {
                            'label': text,
                            'start_time': round(xmin, 4),
                            'end_time': round(xmax, 4),
                            'duration': round(xmax - xmin, 4)
                        }
                        alignments[current_tier].append(entry)
                except:
                    pass

            i += 1

        return alignments
    except Exception as e:
        print(f"Error: {str(e)}")
        return None

# Extract all alignments
all_data = {}
total_words = 0
total_phones = 0

results_dir = '/root/MFA_output'

print("\nExtracting alignments...")
for textgrid_file in Path(results_dir).glob('*.TextGrid'):
    file_id = textgrid_file.stem
    alignments = parse_textgrid_file(str(textgrid_file))

    if alignments:
        all_data[file_id] = alignments
        total_words += len(alignments['words'])
        total_phones += len(alignments['phones'])
        print(f"  ✓ {file_id}")
        print(f"    - Words: {len(alignments['words'])}")
        print(f"    - Phones: {len(alignments['phones'])}")

print(f"\n✓ Total extractions: {len(all_data)} files")
print(f"✓ Total word boundaries: {total_words}")
print(f"✓ Total phone boundaries: {total_phones}")

# Display sample data
if all_data:
    first_file = list(all_data.keys())[0]
    print(f"\n Sample alignments from {first_file}:")
    print("\nWords:")
    for word in all_data[first_file]['words'][:5]:
        print(f"  {word}")
    print("\nPhones:")
    for phone in all_data[first_file]['phones'][:5]:
        print(f"  {phone}")


Extracting alignments...
  ✓ speaker1_utt3
    - Words: 4
    - Phones: 15

✓ Total extractions: 1 files
✓ Total word boundaries: 4
✓ Total phone boundaries: 15

 Sample alignments from speaker1_utt3:

Words:
  {'label': '6', 'start_time': 0.0, 'end_time': 3.0, 'duration': 3.0}
  {'label': 'montreal', 'start_time': 0.03, 'end_time': 1.1, 'duration': 1.07}
  {'label': 'forced', 'start_time': 1.13, 'end_time': 1.28, 'duration': 0.15}
  {'label': 'aligner', 'start_time': 1.28, 'end_time': 2.93, 'duration': 1.65}

Phones:
  {'label': '17', 'start_time': 0.0, 'end_time': 3.0, 'duration': 3.0}
  {'label': 'M', 'start_time': 0.03, 'end_time': 0.06, 'duration': 0.03}
  {'label': 'AH2', 'start_time': 0.06, 'end_time': 0.09, 'duration': 0.03}
  {'label': 'N', 'start_time': 0.09, 'end_time': 0.12, 'duration': 0.03}
  {'label': 'T', 'start_time': 0.12, 'end_time': 0.13, 'duration': 0.01}


In [3]:
import pandas as pd


# Create words dataframe
words_data = []
for file_id, data in all_data.items():
    for word_idx, word in enumerate(data['words']):
        words_data.append({
            'file_id': file_id,
            'word_index': word_idx,
            'word': word['label'],
            'start_time': word['start_time'],
            'end_time': word['end_time'],
            'duration': word['duration']
        })

words_df = pd.DataFrame(words_data)

# Create phones dataframe
phones_data = []
for file_id, data in all_data.items():
    for phone_idx, phone in enumerate(data['phones']):
        phones_data.append({
            'file_id': file_id,
            'phone_index': phone_idx,
            'phone': phone['label'],
            'start_time': phone['start_time'],
            'end_time': phone['end_time'],
            'duration': phone['duration']
        })

phones_df = pd.DataFrame(phones_data)

print(f"\n✓ Words DataFrame shape: {words_df.shape}")
print(f"✓ Phones DataFrame shape: {phones_df.shape}")

print("\n SAMPLE WORD ALIGNMENTS:")
print(words_df.to_string(index=False))

print("\n SAMPLE PHONE ALIGNMENTS:")
print(phones_df.to_string(index=False))


✓ Words DataFrame shape: (4, 6)
✓ Phones DataFrame shape: (15, 6)

 SAMPLE WORD ALIGNMENTS:
      file_id  word_index     word  start_time  end_time  duration
speaker1_utt3           0        6        0.00      3.00      3.00
speaker1_utt3           1 montreal        0.03      1.10      1.07
speaker1_utt3           2   forced        1.13      1.28      0.15
speaker1_utt3           3  aligner        1.28      2.93      1.65

 SAMPLE PHONE ALIGNMENTS:
      file_id  phone_index phone  start_time  end_time  duration
speaker1_utt3            0    17        0.00      3.00      3.00
speaker1_utt3            1     M        0.03      0.06      0.03
speaker1_utt3            2   AH2        0.06      0.09      0.03
speaker1_utt3            3     N        0.09      0.12      0.03
speaker1_utt3            4     T        0.12      0.13      0.01
speaker1_utt3            5     R        0.13      0.16      0.03
speaker1_utt3            6   IY0        0.16      0.19      0.03
speaker1_utt3            

In [4]:


print("\n WORD ALIGNMENT STATISTICS:")
print(f"  Total words: {len(words_df)}")
print(f"  Unique words: {words_df['word'].nunique()}")
print(f"  Mean word duration: {words_df['duration'].mean():.4f} seconds")
print(f"  Median word duration: {words_df['duration'].median():.4f} seconds")
print(f"  Min word duration: {words_df['duration'].min():.4f} seconds")
print(f"  Max word duration: {words_df['duration'].max():.4f} seconds")

print("\n PHONE ALIGNMENT STATISTICS:")
print(f"  Total phones: {len(phones_df)}")
print(f"  Unique phones: {phones_df['phone'].nunique()}")
print(f"  Mean phone duration: {phones_df['duration'].mean():.4f} seconds")
print(f"  Median phone duration: {phones_df['duration'].median():.4f} seconds")
print(f"  Min phone duration: {phones_df['duration'].min():.4f} seconds")
print(f"  Max phone duration: {phones_df['duration'].max():.4f} seconds")

print("\n WORD FREQUENCY:")
print(words_df['word'].value_counts())

print("\n📊 PHONE FREQUENCY:")
print(phones_df['phone'].value_counts())


 WORD ALIGNMENT STATISTICS:
  Total words: 4
  Unique words: 4
  Mean word duration: 1.4675 seconds
  Median word duration: 1.3600 seconds
  Min word duration: 0.1500 seconds
  Max word duration: 3.0000 seconds

 PHONE ALIGNMENT STATISTICS:
  Total phones: 15
  Unique phones: 13
  Mean phone duration: 0.3913 seconds
  Median phone duration: 0.0300 seconds
  Min phone duration: 0.0100 seconds
  Max phone duration: 3.0000 seconds

 WORD FREQUENCY:
word
6           1
montreal    1
forced      1
aligner     1
Name: count, dtype: int64

📊 PHONE FREQUENCY:
phone
T      2
R      2
17     1
AH2    1
M      1
N      1
IY0    1
AA1    1
L      1
F      1
AO1    1
S      1
spn    1
Name: count, dtype: int64
